# 4 Updating data frames

In this shortened version, students were asked to read the sections about adding and removing columns.

This notebook starts from Updating values section.

## Updating values in a dataframe

Start by importing packages and data:

In [1]:
import numpy as np
import pandas as pd

# Create a random number generator with a seed so that work is reproducible
rng = np.random.default_rng(seed=42)

# Import data
URL = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
penguins = pd.read_csv(URL)

# Add new column body_mass_kg 
penguins['body_mass_kg'] = penguins['body_mass_g']/1000

# Create random 3-digit codes
codes = rng.choice(np.arange(100, 1000), size=len(penguins), replace=False)  # Sampling w/o replacement

# Insert codes at the front of data frame
penguins.insert(loc=0,  # Index
                column='id_code',
                value=codes)
        
penguins.head()

,id_code,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,body_mass_kg
0,630,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,3.75
1,679,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,3.80
2,175,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,3.25
3,330,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007,NaN
4,355,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,3.45


For our notes we will need to create id codes for each penguin.

Sometimes we want to update a specific value in our data frame. 

### A single value
Access a single value in a `pandas.DataFrame` using the locators

- `at[]` select by labes

```python
df.at[single_index_value, 'col_name']
```

Think of `at[]` as the equivalent to `loc[]` when trying to access a single value.

#### Example
First update the index of the data frame to be the `id_code` column:

In [4]:
penguins = penguins.set_index('id_code')
penguins.head(3)

KeyError: "None of ['id_code'] are in the columns"

What was the bill length of the penguin with ID number 330? 
We can directly access that information using `at[]`:

In [5]:
# Check bill length of penguin with ID 330
penguins.at[330, 'bill_length_mm']

nan

Update this value to 38.3 mm. We can do this with `at[]` too:

In [6]:
# Correct bill length for penguin with ID 330
penguins.at[330, 'bill_length_mm'] = 38.3

# Confirm value updated
penguins.loc[330]

species                 Adelie
island               Torgersen
bill_length_mm            38.3
bill_depth_mm              NaN
flipper_length_mm          NaN
body_mass_g                NaN
sex                        NaN
year                      2007
body_mass_kg               NaN
Name: 330, dtype: object

If we want to access or update a single value by position we use the `iat[]` locator:

- `iat[]` select by position

```python
df.iat[index_indeger_location, col_integer_location]
```

We can dynamically get the location of a single column in this way:

```python
df.columns.get_loc('column_name')
```

#### Example
Get the absolute position of the column `bill_length_mm`.

In [8]:
penguins.columns.get_loc('bill_length_mm') # 3rd coumn (0,1,2)

2

### Check-in
Use `iat[]` to access the same bill length value for the penguin with ID 330 and revert it back to an NA. Confirm your update using `iloc[]`.

HINT: What would be an equivalent method to penguins.columns.get_loc('column_name') to get the location of a specific label in the index?

In [21]:
penguins.index.get_loc(330)
penguins.iat[3,2] = np.nan # Hard coding row, column. Could have introduced variables row_index and col_index to previous 2 lines of code
penguins.iloc[3]

species                 Adelie
island               Torgersen
bill_length_mm             NaN
bill_depth_mm              NaN
flipper_length_mm          NaN
body_mass_g                NaN
sex                        NaN
year                      2007
body_mass_kg               NaN
Name: 330, dtype: object

### Multiple values in a column

What if we want to update multiple values in a column? 

#### Using a condition 

When we need to create a new column where the new values depend on conditions on another column. 

#### Example

We want to classify the Palmer penguins such that :

- penguins with body mass < 3kg are small, 
- penguins with 3 kg <= body mass < 5 kg are medium, 
- and those with 5 kg <= body mass are large. 

One way to add this information in a new column is with `numpy.select()`:

In [24]:
# Create a list of conditions
conditions = [penguins['body_mass_kg']<3,
                (3<=penguins['body_mass_kg']) & (penguins['body_mass_kg']<5),
                5<=penguins['body_mass_kg']
             ]
# Create a list with the choices
choices = ['small','medium','large']

# Add the selections using np.select
penguins['size'] = np.select(conditions, choices, default=None) # Value for anything outside conditions

# Check
penguins.head()

Signature: np.select(condlist, choicelist, default=0)
Docstring:
Return an array drawn from elements in choicelist, depending on conditions.

Parameters
----------
condlist : list of bool ndarrays
    The list of conditions which determine from which array in `choicelist`
    the output elements are taken. When multiple conditions are satisfied,
    the first one encountered in `condlist` is used.
choicelist : list of ndarrays
    The list of arrays from which the output elements are taken. It has
    to be of the same length as `condlist`.
default : scalar, optional
    The element inserted in `output` when all conditions evaluate to False.

Returns
-------
output : ndarray
    The output at position m is the m-th element of the array in
    `choicelist` where the m-th element of the corresponding array in
    `condlist` is True.

See Also
--------
where : Return elements from one of two arrays depending on condition.
take, choose, compress, diag, diagonal

Examples
--------
>>> x = n

#### By selecting values

We can update some values in a column by selecting this data using `loc` (if selecting by labels) or `iloc` (if selecting by position). The general syntax for updating data with `loc` is:

```python
df.loc[row_selection, col_name] = name_values
```

where 

- `row_selection` is the rows we want to update, this can be any expression that gives us a boolean `pandas.Series`, 
- `col_name` is a single column name, and 
- `new_values` is the new value or values we want. If using multiple values, then `new_values` must be of the same length as the number of rows selected. 

Using `loc[]` in assignment modifies the df directly without need for reasignment

#### Example

Update the "male" values in the sex column to "M".

In [29]:
# Select rows with sex = male and simplify values in sex col
penguins.loc[penguins['sex']=='male','sex'] = 'M'
print(penguins['sex'].unique())

['M' 'female' nan]


In [31]:
# Select rows where sex = female
penguins.loc[penguins['sex']=='female','sex'] = 'F'
print(penguins['sex'].unique())

['M' 'F' nan]


**Avoid chained indexing `[][]` and use `.loc[]` instead**
The `SettingWithCopyWarning` often arises from chained indexing:
```python
df[df['col'] == value]['col2'] = new_value
```

>In the words of the [`pandas` documentation](https://pandas.pydata.org/pandas-docs/version/2.2/user_guide/indexing.html#why-does-assignment-fail-when-using-chained-indexing):

> assigning to the product of chained indexing has inherently unpredictable results.

This warning comes up because some `pandas` operations return a view to your data, while others return a copy of your data.

- **Views** are actual subsets of the original data, when we update them, we are modifying the original data frame. 

- **Copies** are unique objects, independent of our original data frames. When we update a copy we are not modifying the original data frame. 

**GO TO DIAGRAMS**

*Depending on what we are trying to do we might want to modify the original data frame or we might want to modify a copy.*

#### Example

We only want to use data from Biscoe island and, after doing some analyses, we want to add a new column to it:

In [35]:
# Select data from biscoe
biscoe = penguins[penguins['island']=='Biscoe']

# ...other analysis...

# Add a col
biscoe['sample_col'] = 100

/tmp/ipykernel_3441302/2862397482.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  biscoe['sample_col'] = 100


*`pandas` is trying to alert you that it is unsure about whether `biscoe` is a view or a copy and it's unclear whether an of our code will modify our dataset or not.*

To fix this we can **take control of the copy-view situation and explicitly ask for a copy of the dataset when subsetting the data**. 



In [36]:
# Select data from biscoe
biscoe = penguins[penguins['island']=='Biscoe'].copy()

# ...other analysis...

# Add a col
biscoe['sample_col'] = 100

Now we are sure we did not modify our initial data, but rather the `biscoe` data frame:

In [37]:
biscoe.head(3)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,body_mass_kg,size,sample_col
id_code,,,,,,,,,,,
954,Adelie,Biscoe,37.8,18.3,174.0,3400.0,F,2007,3.4,medium,100
503,Adelie,Biscoe,37.7,18.7,180.0,3600.0,M,2007,3.6,medium,100
567,Adelie,Biscoe,35.9,19.2,189.0,3800.0,F,2007,3.8,medium,100
